# 제2장. 손오공은 이름이 몇 개일까? — 읽을 수 있는 자료 만들기

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITPainter/korea-china-classics-text-mining/blob/main/notebooks/ch2_corpus.ipynb)

『서유기』 100회에서 손오공은 몇 번 나올까요? 이름을 한 가지만 찾으면 대부분을 놓칩니다. 컴퓨터가 읽을 수 있는 자료를 만드는 과정을 직접 해 봅니다.

In [1]:
# 실습 환경 준비 (Colab에서 처음 한 번 실행)
import sys
if "google.colab" in sys.modules:
    !pip -q install "kclit @ git+https://github.com/ITPainter/korea-china-classics-text-mining.git"
from kclit.viz import setup_korean_font
print("그래프 글꼴:", setup_korean_font())

그래프 글꼴: AppleGothic


## 2.1 같은 글자가 다르게 보일 때

사람에게는 같은 글자인데 컴퓨터에게는 다른 글자인 경우를 먼저 봅니다.

In [2]:
from kclit import text
for s in ["爲人", "喫人", "吃人", "國破山河在"]:
    print(s, "→ 정규화:", text.normalize(s), "| 간체:", text.to_simplified(s), "| 번체:", text.to_traditional(s))

爲人 → 정규화: 為人 | 간체: 为人 | 번체: 爲人
喫人 → 정규화: 喫人 | 간체: 吃人 | 번체: 喫人
吃人 → 정규화: 吃人 | 간체: 吃人 | 번체: 喫人
國破山河在 → 정규화: 國破山河在 | 간체: 国破山河在 | 번체: 國破山河在


### 「狂人日記」에서 ‘吃人’은 몇 번 나올까요?

루쉰의 위키문헌 판본은 ‘吃人’을 ‘喫人’이라고 적었습니다. 그대로 세어 보고, 간체로 바꾼 뒤 다시 세어 보세요.

In [3]:
from kclit.collect import fetch_wikisource
kuangren = fetch_wikisource("狂人日記", "zh")
print("그대로 — 吃人:", kuangren.count("吃人"), "건 | 喫人:", kuangren.count("喫人"), "건")
print("간체로 바꾼 뒤 — 吃人:", text.to_simplified(kuangren).count("吃人"), "건")

그대로 — 吃人: 0 건 | 喫人: 28 건
간체로 바꾼 뒤 — 吃人: 28 건


## 2.2 원문 내려받기

이 책은 저작권 보호 기간이 끝난 두 고전을 끝까지 따라갑니다. 중국의 『서유기』(백화문)와 한국의 『구운몽』(한문 원문)입니다. 위키문헌에서 받아 오며, 같은 사이트에는 2초 간격을 둡니다.

In [4]:
from kclit import classics
xiyou = classics.xiyou()   # 처음에는 100회를 받느라 4~5분 걸립니다. 한 번 받으면 data 폴더에 저장됩니다.

In [5]:
gu = classics.gu()   # 『구운몽』 16회
print("서유기:", len(xiyou), "회,", sum(len(c) for c in xiyou), "자")
print("구운몽:", len(gu), "회,", sum(len(c) for c in gu), "자")
print(xiyou[0][:120])

서유기: 100 회, 721762 자
구운몽: 16 회, 80783 자
詩曰：

混沌未分天地亂，茫茫渺渺無人見。
自從盤古破鴻濛，開闢從茲清濁辨。
覆載群生仰至仁，發明萬物皆成善。
欲知造化會元功，須看《西遊釋厄傳》。
蓋聞天地之數，有十二萬九千六百歲為一元。將一元分為十二會，乃子、丑、寅、卯、辰、巳、午、未


## 2.3 단어 나누기

한국어는 Kiwi로 형태소를 나누고, 중국어는 jieba로 단어를 나눕니다. 작품 속 이름은 사용자 사전에 등록해야 합니다.

In [6]:
from kclit import tokenize
print(tokenize.korean("손오공은 삼장법사를 따라 서천으로 떠났다"))
line = "孫行者道：「師父，我們走罷。」"
print("등록 전:", tokenize.chinese(line))
tokenize.add_chinese_words(["孫行者"])
print("등록 후:", tokenize.chinese(line))

['손오공', '삼장', '법사', '따르다', '서천', '떠나다']


등록 전: ['孫', '行者', '道', '師父', '我們', '走罷']
등록 후: ['孫行者', '道', '師父', '我們', '走罷']


## 2.4 호칭 사전과 이름표

회차마다 작가, 시대, 장르 같은 이름표를 붙여 코퍼스 표로 만듭니다. 빠진 칸이 있는지도 점검합니다.

In [7]:
from kclit import corpus
df = classics.corpus_frame(xiyou, gu)
print(corpus.check_metadata(df))     # 창작 연도는 학설에 따라 달라 비워 두었습니다
corpus.distribution(df, ["country", "script", "title"]).head()

['year: 빈 칸 116개']


,country,script,title,작품 수
0,중국,백화문,西遊記 第100回,1
1,중국,백화문,西遊記 第10回,1
2,중국,백화문,西遊記 第11回,1
3,중국,백화문,西遊記 第12回,1
4,중국,백화문,西遊記 第13回,1


### 사람에게도 이름표를: 호칭 사전

『서유기』에서 ‘孫悟空’만 검색한 결과와, 손오공을 가리키는 모든 호칭을 합친 결과를 비교합니다.

In [8]:
from kclit import people
print("孫悟空만 검색:", sum(c.count("孫悟空") for c in xiyou), "번")
counts = people.count_names(xiyou, people.XIYOU_GROUPS)
print("모든 호칭 합계:", int(counts["손오공"].sum()), "번")
people.alias_table(xiyou, people.XIYOU_GROUPS["손오공"])

孫悟空만 검색: 126 번
모든 호칭 합계: 7036 번


,횟수,비중
行者,4094,0.582
大聖,959,0.136
老孫,618,0.088
悟空,381,0.054
孫行者,230,0.033
孫大聖,202,0.029
猴王,139,0.020
孫悟空,126,0.018
齊天大聖,103,0.015
猴子,81,0.012


In [9]:
# ‘行者’는 언제부터 쓰였을까? (‘孫行者’는 따로 센 호칭이라 단독 ‘行者’만 셉니다)
import re
pattern = re.compile("孫行者|行者")
early = sum(m.group() == "行者" for c in xiyou[:13] for m in pattern.finditer(c))
late = sum(m.group() == "行者" for c in xiyou[13:] for m in pattern.finditer(c))
print("1~13회:", early, "번 | 14~100회:", late, "번 | 합계:", early + late)

1~13회: 10 번 | 14~100회: 4084 번 | 합계: 4094


호칭이 13가지나 되고, 가장 많이 쓰이는 것은 ‘行者’입니다. 사람이 호칭 사전을 만들어 주지 않으면 컴퓨터는 한 인물을 여러 사람으로 읽습니다. 3장에서 이 사전을 그대로 씁니다.